# 03 — Modeling, Evaluation & Hyperparameter Tuning
**Loan Default Prediction Project**

Goal: train L1 and L2 logistic regression, evaluate thoroughly, tune the decision threshold, run GridSearchCV, and save the final model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings, sys, os

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, brier_score_loss, ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))
from src.features import add_features

RANDOM_STATE = 42
PLOTS_DIR = '../plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Load Preprocessing Artifacts

In [ ]:
X_train_fe, X_test_fe, y_train, y_test, preprocessor, all_features = \
    joblib.load('../models/preprocessing_artifacts.pkl')

print(f'Train: {X_train_fe.shape}  |  Test: {X_test_fe.shape}')
print(f'Features: {len(all_features)}')

## 2. Baseline: DummyClassifier

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train_fe, y_train)
y_pred_dummy = dummy.predict(X_test_fe)

print('=== Dummy Classifier (Baseline) ===')
print(classification_report(y_test, y_pred_dummy, target_names=['No Default','Default']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_dummy):.4f}  (this is your floor)')

## 3. Logistic Regression — L2 (Ridge)

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
pre = ColumnTransformer([('num', numeric_transformer, all_features)])

lr_l2 = Pipeline([
    ('preprocessor', pre),
    ('classifier', LogisticRegression(
        C=1.0, penalty='l2', solver='lbfgs',
        class_weight='balanced',
        max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1
    ))
])

lr_l2.fit(X_train_fe, y_train)
print('L2 model trained.')

In [ ]:
y_pred_l2       = lr_l2.predict(X_test_fe)
y_pred_proba_l2 = lr_l2.predict_proba(X_test_fe)[:, 1]

print('=== Logistic Regression — L2 ===')
print(classification_report(y_test, y_pred_l2, target_names=['No Default','Default']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_proba_l2):.4f}')

## 4. Logistic Regression — L1 (Lasso / Feature Selection)

In [ ]:
pre2 = ColumnTransformer([('num', Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
]), all_features)])

lr_l1 = Pipeline([
    ('preprocessor', pre2),
    ('classifier', LogisticRegression(
        C=1.0, penalty='l1', solver='saga',
        class_weight='balanced',
        max_iter=2000, random_state=RANDOM_STATE, n_jobs=-1
    ))
])

lr_l1.fit(X_train_fe, y_train)
y_pred_proba_l1 = lr_l1.predict_proba(X_test_fe)[:, 1]
print(f'L1 ROC-AUC: {roc_auc_score(y_test, y_pred_proba_l1):.4f}')

# Which features did L1 zero out?
coefs_l1 = lr_l1.named_steps['classifier'].coef_[0]
zeroed   = [f for f,c in zip(all_features, coefs_l1) if abs(c) < 1e-6]
print(f'Features zeroed by L1: {zeroed if zeroed else "None — all survived"} ')

## 5. ROC & Precision-Recall Curves

In [ ]:
fpr_l2, tpr_l2, _  = roc_curve(y_test, y_pred_proba_l2)
fpr_l1, tpr_l1, _  = roc_curve(y_test, y_pred_proba_l1)
auc_l2 = roc_auc_score(y_test, y_pred_proba_l2)
auc_l1 = roc_auc_score(y_test, y_pred_proba_l1)

prec_l2, rec_l2, thresh_pr = precision_recall_curve(y_test, y_pred_proba_l2)
ap_l2 = average_precision_score(y_test, y_pred_proba_l2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr_l2, tpr_l2, color='#185FA5', lw=2, label=f'L2 (AUC={auc_l2:.4f})')
axes[0].plot(fpr_l1, tpr_l1, color='#0F6E56', lw=2, ls='--', label=f'L1 (AUC={auc_l1:.4f})')
axes[0].plot([0,1],[0,1],'k--',lw=1,label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

# PR
axes[1].plot(rec_l2, prec_l2, color='#993C1D', lw=2, label=f'L2 (AP={ap_l2:.4f})')
axes[1].axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/06_roc_pr_curves.png')
plt.show()

## 6. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_l2, ax=ax,
    display_labels=['No Default','Default'],
    cmap='Blues', colorbar=False
)
ax.set_title('Confusion Matrix — L2 Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/07_confusion_matrix.png')
plt.show()

## 7. Threshold Tuning

In [ ]:
thresholds = np.arange(0.05, 0.90, 0.01)
f1_scores  = [f1_score(y_test,(y_pred_proba_l2>=t).astype(int), pos_label=1) for t in thresholds]
rec_scores = []
for t in thresholds:
    preds = (y_pred_proba_l2 >= t).astype(int)
    from sklearn.metrics import recall_score
    rec_scores.append(recall_score(y_test, preds, pos_label=1))

best_thresh_f1 = thresholds[np.argmax(f1_scores)]
best_f1        = max(f1_scores)

# Threshold that gives >= 80% recall
target_recall = 0.80
valid_idx = [i for i,r in enumerate(rec_scores) if r >= target_recall]
thresh_80recall = thresholds[max(valid_idx)] if valid_idx else None

print(f'Best threshold for F1:        {best_thresh_f1:.2f}  (F1={best_f1:.4f})')
print(f'Threshold for 80% recall:     {thresh_80recall}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, f1_scores,  color='#185FA5', label='F1 Score (Defaulters)')
ax.plot(thresholds, rec_scores, color='#993C1D', ls='--', label='Recall (Defaulters)')
ax.axvline(best_thresh_f1, color='#185FA5', lw=1.5, ls=':')
if thresh_80recall:
    ax.axvline(thresh_80recall, color='#993C1D', lw=1.5, ls=':')
ax.axhline(0.80, color='gray', lw=0.8, ls='--', label='80% recall target')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/08_threshold_tuning.png')
plt.show()

In [ ]:
# Apply best threshold
BEST_THRESHOLD = best_thresh_f1
y_pred_opt = (y_pred_proba_l2 >= BEST_THRESHOLD).astype(int)
print(f'Using threshold: {BEST_THRESHOLD:.2f}')
print(classification_report(y_test, y_pred_opt, target_names=['No Default','Default']))

## 8. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    lr_l2, X_train_fe, y_train, cv=cv,
    scoring=['roc_auc','average_precision','f1'],
    return_train_score=True, n_jobs=-1
)
print('5-Fold Stratified Cross-Validation Results:')
print('-'*50)
for metric in ['roc_auc','average_precision','f1']:
    tr = cv_results[f'train_{metric}']
    te = cv_results[f'test_{metric}']
    print(f'{metric:25s}: train={tr.mean():.4f}±{tr.std():.4f}  val={te.mean():.4f}±{te.std():.4f}')

## 9. Hyperparameter Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'classifier__C':       [0.001, 0.01, 0.1, 1.0, 10.0],
    'classifier__penalty': ['l1', 'l2'],
}

pre3 = ColumnTransformer([('num', Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
]), all_features)])

lr_base = Pipeline([
    ('preprocessor', pre3),
    ('classifier', LogisticRegression(
        solver='saga', class_weight='balanced',
        max_iter=2000, random_state=RANDOM_STATE, n_jobs=-1
    ))
])

grid_search = GridSearchCV(
    lr_base, param_grid, cv=5,
    scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search.fit(X_train_fe, y_train)

print(f'Best params:  {grid_search.best_params_}')
print(f'Best CV AUC:  {grid_search.best_score_:.4f}')

In [ ]:
# GridSearch results heatmap
results_df = pd.DataFrame(grid_search.cv_results_)
results_df['C'] = results_df['param_classifier__C']
results_df['penalty'] = results_df['param_classifier__penalty']
pivot = results_df.pivot(index='C', columns='penalty', values='mean_test_score')

plt.figure(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('GridSearchCV — ROC-AUC by C and Penalty', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/09_gridsearch_heatmap.png')
plt.show()

## 10. Save Final Model

In [ ]:
best_model = grid_search.best_estimator_
y_pred_best       = best_model.predict(X_test_fe)
y_pred_proba_best = best_model.predict_proba(X_test_fe)[:,1]

final_auc = roc_auc_score(y_test, y_pred_proba_best)
final_ap  = average_precision_score(y_test, y_pred_proba_best)
final_brier = brier_score_loss(y_test, y_pred_proba_best)

print('=== FINAL MODEL PERFORMANCE ===')
print(f'ROC-AUC:          {final_auc:.4f}')
print(f'Avg Precision:    {final_ap:.4f}')
print(f'Brier Score:      {final_brier:.4f}')
print()
print(classification_report(y_test, (y_pred_proba_best>=BEST_THRESHOLD).astype(int),
                            target_names=['No Default','Default']))

joblib.dump(best_model, '../models/loan_default_model.pkl')
print('\nModel saved to models/loan_default_model.pkl')

## Final Results Summary

| Metric | Value |
|--------|-------|
| ROC-AUC | *see above* |
| Average Precision | *see above* |
| Brier Score | *see above* |
| Best C | *see best_params_* |
| Best Penalty | *see best_params_* |
| Decision Threshold | *see BEST_THRESHOLD* |